# Étape 8 – Export vers Fabric (tables Delta pour Power BI)

À lancer après `06c_scenarios` et `07_figures`. Ce notebook ne recalcule rien : il relit les sorties du pipeline
(`outputs/`) et les écrit en **tables Delta** dans le Lakehouse attaché, prêtes pour un modèle sémantique Power BI.

| Table | Grain | Usage Power BI |
|---|---|---|
| `carreaux` | carreau 200 m habité (secret statistique respecté) | carte de l'accès, filtres par commune / décile |
| `communes` | commune | carte choroplèthe, classements |
| `stations` | point d'offre (arrêt, station vélo, Citiz…) | carte de l'offre existante |
| `recommandations` | site proposé × zone étudiée | carte et tableau des sites, coûts cumulés |
| `territoires_citiz` | intercommunalité | classement des territoires pour Citiz |
| `acces_par_mode` | périmètre × mode | part de la population desservie par mode |
| `acces_par_decile` | périmètre × décile de revenu | courbe en U revenu / accès |
| `modele_metriques` | type × modèle × métrique | qualité des modèles |
| `modele_importance` | type × variable | variables explicatives |
| `scenarios_sites` | site × territoire × financeur × budget × λ | carte et liste des sites du scénario choisi |
| `scenarios_resume` | territoire × financeur × budget × λ | KPI du scénario, courbe « prix de l'équité » |
| `dim_zone` | territoire des scénarios | segment unique « territoire » (clés `zone`, `epci`) |
| `dim_financeur` | financeur | segment Citiz / collectivité |
| `param_budget_citiz`, `param_budget_collectivite`, `param_lambda` | valeurs des curseurs | tables déconnectées des paramètres *what-if* |

Clés de relation : `com` (code commune INSEE, texte sur 5 caractères) relie `carreaux`, `communes` et `recommandations` ; `zone` relie `dim_zone` aux tables `scenarios_*` et `recommandations` ; `epci` (SIREN) relie `dim_zone`, `communes` et `territoires_citiz`. Les paramètres sont des entiers (`budget_k_eur`, `lambda_x10`) : un λ décimal (0,3) ne s'égalise pas de façon fiable en DAX.

Les tables passent par un parquet intermédiaire dans `outputs/fabric/` ; hors Fabric, seul ce parquet est écrit (pour tester en local).

In [ ]:
# Fabric : à exécuter en premier. En local (hors Fabric), cette cellule ne fait rien.
import os
import subprocess
import sys

FABRIC_PROJECT = "/lakehouse/default/Files/mobiliscore"  # dossier du projet uploadé dans le Lakehouse
if os.path.isdir("/lakehouse/default"):
    os.chdir(FABRIC_PROJECT)  # la préparation trouve Config.py dans le dossier courant
    # inutile si un Environment Fabric fournit déjà ces paquets (recommandé : voir FABRIC.md)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyproj>=3.6", "openpyxl>=3.1"], check=True)

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "Config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import Config as C  # noqa: E402

try:
    spark  # noqa: F821 – session fournie par les notebooks Fabric
    IN_FABRIC = True
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")  # noqa: F821
except NameError:
    IN_FABRIC = False
IN_LAKEHOUSE = Path("/lakehouse/default/Tables").is_dir()   # notebook Python (sans Spark) : deltalake
LOCAL_OUT = C.OUT / "fabric"
print("Fabric" if IN_FABRIC else f"local -> {LOCAL_OUT}")

In [ ]:
def write_table(df, name):
    """Écrit une table Delta dans le Lakehouse (Fabric) ou un parquet (local), en écrasant l'existant."""
    df = df.reset_index(drop=True)
    for col in df.columns:
        if df[col].dtype == object:  # colonnes texte homogènes (Spark refuse les types mélangés)
            df[col] = df[col].astype("string")
    LOCAL_OUT.mkdir(parents=True, exist_ok=True)
    path = LOCAL_OUT / f"{name}.parquet"
    df.to_parquet(path, index=False)  # pyarrow gère tous les types pandas (entiers nullables, texte…)
    if IN_FABRIC:
        # /lakehouse/default/Files/... côté Python = Files/... côté Spark
        (spark.read.parquet(str(path).replace("/lakehouse/default/", ""))  # noqa: F821
         .write.mode("overwrite").option("overwriteSchema", "true")
         .format("delta").saveAsTable(name))
    elif IN_LAKEHOUSE:
        from deltalake import write_deltalake   # préinstallé dans les notebooks Python de Fabric
        write_deltalake(f"/lakehouse/default/Tables/{name}", df, mode="overwrite", schema_mode="overwrite")
    print(f"{name:<20} {len(df):>9,} lignes  {df.shape[1]:>3} colonnes")


def com_code(s):
    """Code commune INSEE en texte sur 5 caractères (01001, 2A004…)."""
    return s.astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(5)


summary = json.loads((C.OUT / "access_summary.json").read_text(encoding="utf-8"))
MODES = ["rail", "transit", "bike", "scooter", "carsharing", "citiz"]
MODE_LABELS = {"rail": "Train", "transit": "Bus, tram, métro", "bike": "Vélo en libre-service",
               "scooter": "Free-floating", "carsharing": "Autre autopartage", "citiz": "Citiz"}

## Carreaux

Seuls les carreaux d'au moins `PRIVACY_MIN_HOUSEHOLDS` ménages sont exportés (secret statistique INSEE),
comme pour les cartes du projet. Le décile de revenu reprend les seuils nationaux de `access_summary.json`.

In [ ]:
cells = pd.read_parquet(C.OUT / "cells_access.parquet")
n_all = len(cells)
cells = cells[cells["privacy_ok"]].drop(columns=["privacy_ok", "crs_id", "x", "y"])
cells["com"] = com_code(cells["com"])
cells["decile_revenu"] = (np.searchsorted(summary["income_decile_thresholds"], cells["income_pc"], side="right") + 1)
cells.loc[cells["income_pc"].isna(), "decile_revenu"] = np.nan
cells["decile_revenu"] = cells["decile_revenu"].astype("Int8")
cells["sans_option"] = cells["n_options"] == 0
print(f"{n_all - len(cells):,} carreaux sous le seuil de secret statistique écartés")
write_table(cells, "carreaux")

## Communes

In [ ]:
communes = pd.read_parquet(C.OUT / "commune_access.parquet")
communes = communes[communes["privacy_ok"]].drop(columns="privacy_ok")
communes["com"] = com_code(communes["com"])
from src import recommend, territories  # noqa: E402
plm = communes["com"].map(recommend.plm_label)              # arrondissements de Paris, Lyon, Marseille
communes["nom"] = communes["nom"].fillna(plm) if "nom" in communes else plm
communes["epci"] = pd.Series(territories.com_to_epci(communes["com"]), index=communes.index).astype("string")
write_table(communes, "communes")

## Stations (offre existante)

Les populations desservies d'une station sont masquées quand elles touchent moins de `PRIVACY_MIN_HOUSEHOLDS` ménages.

In [ ]:
stations = pd.read_parquet(C.OUT / "stations_served.parquet")
hidden = ~stations["privacy_ok"]
stations.loc[hidden, ["pop_served", "pop_served_exclusive", "households_served"]] = np.nan
stations = stations.drop(columns=["privacy_ok", "crs_id", "x", "y"])
stations["mode_libelle"] = stations["mode"].map(MODE_LABELS).fillna(stations["mode"])
write_table(stations, "stations")

## Recommandations

Toutes les zones calculées (`recommendations*.csv`) dans une seule table, avec la zone en colonne pour la filtrer dans Power BI.

In [ ]:
recos = []
for path in sorted(C.OUT.glob("recommendations*.csv")):
    suffix = path.stem.removeprefix("recommendations")
    meta = json.loads((C.OUT / f"recommendations_summary{suffix}.json").read_text(encoding="utf-8"))
    df = pd.read_csv(path)
    df.insert(0, "zone", meta["area"])
    df.insert(1, "zone_nom", meta["territoire"])
    df.insert(2, "zone_par_defaut", suffix == "")
    recos.append(df)
recos = pd.concat(recos, ignore_index=True)
recos["com"] = com_code(recos.pop("code_commune"))
recos["zone"] = recos["zone"].str.replace(":", "", regex=False)   # même clé que scenarios_* et dim_zone
write_table(recos, "recommandations")

## Territoires Citiz

In [ ]:
terr = pd.read_csv(C.OUT / "territories_citiz.csv", dtype={"epci": "string", "dep": "string"})
write_table(terr, "territoires_citiz")

## Indicateurs d'accès (`access_summary.json`)

Format long : une ligne par périmètre et par mode, ou par périmètre et par décile.

In [ ]:
scopes = {"France": summary["national"], "Eurométropole de Strasbourg": summary["ems"],
          **summary["by_territory"]}
rows = [{"perimetre": scope, "mode": m, "mode_libelle": MODE_LABELS[m], "part_population_desservie": v["share_served_by_mode"][m],
         "population": v["population"], "part_sans_option": v["share_no_option"], "score_moyen": v["score_mean"]}
        for scope, v in scopes.items() for m in MODES]
write_table(pd.DataFrame(rows), "acces_par_mode")

deciles = {"France": "by_income_decile", "France (secret statistique)": "by_income_decile_privacy_ok",
           "Strasbourg (déciles nationaux)": "ems_by_income_decile_national",
           "Strasbourg (déciles locaux)": "ems_by_income_decile_local"}
rows = []
for scope, key in deciles.items():
    for dec, v in summary[key].items():
        row = {"perimetre": scope, "decile_revenu": int(dec), "population": v["population"],
               "part_sans_option": v["share_no_option"], "n_options_moyen": v["n_options_mean"],
               "score_moyen": v["score_mean"], "revenu_min": v["income_pc_range"][0], "revenu_max": v["income_pc_range"][1]}
        row.update({f"part_{m}": share for m, share in v.get("share_served_by_mode", {}).items()})
        rows.append(row)
write_table(pd.DataFrame(rows), "acces_par_decile")

## Modèles (`model_metrics.json`)

In [ ]:
metrics = json.loads((C.OUT / "model_metrics.json").read_text(encoding="utf-8"))
rows, imp = [], []
for kind, k in metrics["kinds"].items():
    for model, res in k["models"].items():
        for agg, vals in res.items():
            for metric, value in vals.items():
                rows.append({"type": kind, "modele": model, "retenu": model == k["selected_model"],
                             "agregation": agg, "metrique": metric, "valeur": value})
    imp += [{"type": kind, "variable": f, "importance": v} for f, v in k["permutation_importance"].items()]
write_table(pd.DataFrame(rows), "modele_metriques")
write_table(pd.DataFrame(imp), "modele_importance")

## Scénarios des curseurs (`06c_scenarios`)

Grille précalculée par `src/scenarios.py` (financeur × budget × λ, territoire par territoire). Les tables de
dimension et de paramètres évitent de créer des tables calculées DAX (limitées en mode Direct Lake).

In [ ]:
res = pd.read_parquet(C.OUT / "fabric" / "scenarios_resume.parquet")
sites = pd.read_parquet(C.OUT / "fabric" / "scenarios_sites.parquet")
sites["com"] = com_code(sites["com"])
write_table(res, "scenarios_resume")
write_table(sites, "scenarios_sites")

dim_zone = (res[["zone", "epci", "zone_nom", "rang_citiz", "ordre", "zone_par_defaut"]]
            .drop_duplicates("zone").sort_values("ordre"))
write_table(dim_zone, "dim_zone")
write_table(pd.DataFrame({"financeur": ["citiz", "collectivite"],
                          "libelle": ["Citiz (stations d'autopartage)", "Collectivité (vélo, bus)"]}), "dim_financeur")
b = np.sort(res["budget_k_eur"].unique())
pb = pd.DataFrame({"budget_k_eur": b, "budget_meur": b / 1000})
write_table(pb, "param_budget_citiz")            # un curseur par financeur
write_table(pb, "param_budget_collectivite")
lx = np.sort(res["lambda_x10"].unique())
write_table(pd.DataFrame({"lambda_x10": lx, "lambda": lx / 10}), "param_lambda")

## Ensuite, dans Fabric

1. Ouvrir le Lakehouse : les tables apparaissent sous **Tables** (actualiser si besoin).
2. **New semantic model** et cocher les tables.
3. Relations un-à-plusieurs : `communes[com]` → `carreaux[com]` ; `dim_zone[zone]` → `scenarios_resume[zone]`,
   `scenarios_sites[zone]`, `recommandations[zone]` ; `dim_financeur[financeur]` → ces trois mêmes tables ;
   `dim_zone[epci]` → `communes[epci]`. `param_budget_*` et `param_lambda` restent **déconnectées** (curseurs).
4. Mesures DAX et pages du rapport : voir FABRIC.md (section Power BI).